# CITADEL Single-Notebook Experiment Runner

This notebook is the one-stop runner for the CITADEL journal-extension workspace. It locates the tracked telemetry data, reproduces the EXACT baseline, runs the CITADEL/TCAD ablation grid, attaches hardware-cost estimates, exports FPGA/RTL golden vectors, and shows how future RTL synthesis results can be merged back into the paper tables.

The default settings use the tracked DDR telemetry in `data/telemetry/processed/ddr_data/` with the smoke grid. For final TCAD numbers, keep `DATA_MODE = "real"` and set `TCAD_PRESET` to `"full"`. Apple tier data is located at `data/telemetry/raw/apple_data/` and is kept separate from CINTAS hardware-cost claims.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

EARLY_SEED = "123"
EARLY_THREADS = "1"
for key in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[key] = EARLY_THREADS
os.environ["PYTHONHASHSEED"] = EARLY_SEED
os.environ["MPLBACKEND"] = "Agg"
_mpl_config_dir = Path(tempfile.gettempdir()) / "exact-matplotlib"
_mpl_config_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(_mpl_config_dir)

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "exact").is_dir():
            return candidate
    raise RuntimeError(f"Could not find CITADEL repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")
print(f"Python: {platform.python_version()} on {platform.platform()}")


## 1. Reproducibility Configuration

The notebook uses deterministic settings for every experiment:

- seed: `123`
- numerical threads: `1`
- automatic DDR and Apple telemetry discovery
- deterministic sample-data generator when `DATA_MODE = "sample"`
- repo-relative paths
- run manifests with package versions, git commit, input hashes, and output hashes

For the strictest cross-machine comparison, start Jupyter itself with `PYTHONHASHSEED=123`. The notebook still sets the environment variable for downstream tools.


In [ ]:
SEED = 123
THREADS = 1
SAMPLE_ROWS = 600

DDR_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "processed" / "ddr_data"
APPLE_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "raw" / "apple_data"
REAL_DATA_ROOT = DDR_DATA_ROOT
APPLE_TIER_DATA_ROOT = APPLE_DATA_ROOT
DATA_SOURCE_CONFIG = REPO_ROOT / "data" / "external_sources.json"

def csv_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))


def is_git_lfs_pointer(path: Path) -> bool:
    try:
        with path.open("rb") as f:
            return f.read(64).startswith(b"version https://git-lfs.github.com/spec")
    except OSError:
        return False


DDR_CSVS = csv_files(DDR_DATA_ROOT)
APPLE_CSVS = csv_files(APPLE_DATA_ROOT)
pointer_files = [p for p in [*DDR_CSVS[:3], *APPLE_CSVS[:3]] if is_git_lfs_pointer(p)]
if pointer_files:
    listed = "\n".join(str(p.relative_to(REPO_ROOT)) for p in pointer_files[:6])
    raise RuntimeError(
        "Telemetry CSVs are still Git LFS pointer files. Fetch the Git LFS objects "
        "with your Git client before running the notebook. Examples:\n" + listed
    )

# The repo now tracks real telemetry. Change to "sample" only for a tiny pipeline check.
DATA_MODE = "real" if DDR_CSVS else "sample"  # "real" or "sample"
TCAD_PRESET = "smoke"  # "smoke" for laptop/debug, "full" for journal-scale sweeps
RUN_REPEAT_CHECK = True

RESULTS_ROOT = REPO_ROOT / "results" / "notebook_run"
DATA_ROOT = REPO_ROOT / "data" / "sample" if DATA_MODE == "sample" else REAL_DATA_ROOT
ETS_OUT = RESULTS_ROOT / "ets_baseline"
TCAD_OUT = RESULTS_ROOT / "tcad_ablation"
TCAD_REPEAT_OUT = RESULTS_ROOT / "tcad_ablation_repeat"
FPGA_OUT = RESULTS_ROOT / "fpga"
RTL_SWEEP_OUT = RESULTS_ROOT / "rtl_sweep"

from exact.repro import configure_reproducibility

env_updates = configure_reproducibility(seed=SEED, threads=THREADS, matplotlib_backend="Agg")
for key, value in env_updates.items():
    print(f"{key}={value}")

## 2. Integrated Notebook Utilities

This section replaces the former standalone utility files. Keep these functions in the notebook so data preparation, sample-data generation, baseline reproduction, TCAD ablation, figure generation, and manifest handling all run from one place.


In [ ]:
import hashlib
import json
import os
import subprocess
import time
import urllib.parse
import urllib.request
from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd

from exact.causal_corr import build_causal_and_rank_features_for_setup, export_edges_for_fig5
from exact.cintas import CINTASModel, FixedPointCINTAS, FixedPointConfig, fit_cintas_from_benign
from exact.evaluation import run_exact_eval_for_setup, save_exact_summaries_for_setup
from exact.fig5 import plot_benign_sparse_causal_network_two_panel
from exact.fig6 import plot_topk_features_two_panel
from exact.hardware import estimate_cintas_hardware_cost, format_feature_group_counts
from exact.io import load_telemetry_two_setups
from exact.plotting import plot_metrics_vs_window_size
from exact.preprocessing import clean_and_debias_telemetry, drop_constant_features, get_feature_columns
from exact.repro import find_repo_root as exact_find_repo_root, write_run_manifest
from exact.sample_data import create_sample_dataset


# ---- EXACT baseline reproduction, now owned by this notebook ----




@dataclass(frozen=True)
class ETS2026Config:
    scenarios_eval: tuple[str, ...] = ("DROOP", "RH", "SPECTRE")
    window_sizes: tuple[int, ...] = tuple(range(50, 1001, 50))
    n_splits: int = 5
    lambda_res: float = 0.5
    p_quantile: float = 0.99
    agg_mode: str = "max"
    corr_threshold: float = 0.35
    top_k_features: int = 15
    seed: int = 123


def run_ets2026(
    *,
    data_root: Path,
    out_root: Path,
    cfg: ETS2026Config = ETS2026Config(),
) -> dict:
    """Run the (notebook-derived) EXACT pipeline end-to-end.

    Outputs
    -------
    A dictionary with key artifacts:
      - 'setup_A', 'setup_B': loaded telemetry DataFrames
      - 'shared_features': list[str]
      - 'model_A', 'model_B': fitted CINTAS models
      - 'results_A', 'results_B': evaluation fold-level DataFrames
      - 'ranks_A', 'ranks_B': feature rank DataFrames
    """
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    repo_root = REPO_ROOT

    # ------------------------------------------------------------------
    # Load
    # ------------------------------------------------------------------
    df_A, df_B = load_telemetry_two_setups(Path(data_root))

    # ------------------------------------------------------------------
    # Clean + debias (matches EXACT.ipynb)
    # ------------------------------------------------------------------
    df_A_z = clean_and_debias_telemetry(df_A)
    df_B_z = clean_and_debias_telemetry(df_B)

    feat_A = drop_constant_features(df_A_z, get_feature_columns(df_A_z))
    feat_B = drop_constant_features(df_B_z, get_feature_columns(df_B_z))
    shared_features = sorted(set(feat_A) & set(feat_B))

    if not shared_features:
        raise RuntimeError("No shared numeric telemetry features between Setup A and Setup B.")

    # ------------------------------------------------------------------
    # Fit CINTAS from BENIGN (same feature list for A and B)
    # ------------------------------------------------------------------
    model_A = fit_cintas_from_benign(df_A_z, shared_features, lambda_res=cfg.lambda_res)
    model_B = fit_cintas_from_benign(df_B_z, shared_features, lambda_res=cfg.lambda_res)

    # Attach per-sample score column for later ranking
    for df_z, model in [(df_A_z, model_A), (df_B_z, model_B)]:
        score, _, _ = model.score_dataframe(df_z)
        df_z["cias_sample_score"] = score

    # ------------------------------------------------------------------
    # Offline causal correlation skeleton + feature ranking
    # ------------------------------------------------------------------
    nodes_A, edges_A, ranks_A = build_causal_and_rank_features_for_setup(
        setup="A",
        df=df_A_z,
        feature_cols=shared_features,
        out_root=out_root,
        corr_threshold=cfg.corr_threshold,
        top_k_plot=20,
        score_col="cias_sample_score",
    )
    nodes_B, edges_B, ranks_B = build_causal_and_rank_features_for_setup(
        setup="B",
        df=df_B_z,
        feature_cols=shared_features,
        out_root=out_root,
        corr_threshold=cfg.corr_threshold,
        top_k_plot=20,
        score_col="cias_sample_score",
    )

    # Export edges in Fig.5-compatible format (optional convenience)
    export_edges_for_fig5(edges_A, out_root / "fig5_inputs" / "setupA_edges.csv")
    export_edges_for_fig5(edges_B, out_root / "fig5_inputs" / "setupB_edges.csv")

    # Fig. 6-style feature importance plot (from rank CSVs)
    plot_topk_features_two_panel(
        ranks_csv_A=out_root / "causal" / "SETUP_A_feature_ranks.csv",
        ranks_csv_B=out_root / "causal" / "SETUP_B_feature_ranks.csv",
        out_png=out_root / "figures" / "fig6_top15_features.png",
        top_k=cfg.top_k_features,
    )

    # ------------------------------------------------------------------
    # Evaluation (fold-level)
    # ------------------------------------------------------------------
    results_A = run_exact_eval_for_setup(
        setup="A",
        df=df_A_z,
        model=model_A,
        scenarios_eval=cfg.scenarios_eval,
        window_sizes=cfg.window_sizes,
        n_splits_list=(cfg.n_splits,),
        default_p_quantile=cfg.p_quantile,
        agg_mode=cfg.agg_mode,
        droop_cfg=None,
        seed=cfg.seed,
    )
    results_B = run_exact_eval_for_setup(
        setup="B",
        df=df_B_z,
        model=model_B,
        scenarios_eval=cfg.scenarios_eval,
        window_sizes=cfg.window_sizes,
        n_splits_list=(cfg.n_splits,),
        default_p_quantile=cfg.p_quantile,
        agg_mode=cfg.agg_mode,
        droop_cfg=None,
        seed=cfg.seed,
    )

    # Save summaries (these CSV names match EXACT.ipynb plots)
    if not results_A.empty:
        save_exact_summaries_for_setup("A", results_A, out_root)
    if not results_B.empty:
        save_exact_summaries_for_setup("B", results_B, out_root)

    # Plot metrics-vs-window-size from the per-workload CSVs
    per_wl_A = out_root / "SETUP_A_EXACT_summary_per_workload.csv"
    per_wl_B = out_root / "SETUP_B_EXACT_summary_per_workload.csv"
    if per_wl_A.exists() and per_wl_B.exists():
        plot_metrics_vs_window_size(
            per_workload_csv_A=per_wl_A,
            per_workload_csv_B=per_wl_B,
            out_png=out_root / "figures" / "fig4_metrics_vs_window_size.png",
            n_splits=cfg.n_splits,
        )

    artifact_paths = (
        out_root / "SETUP_A_EXACT_summary_global.csv",
        out_root / "SETUP_A_EXACT_summary_per_workload.csv",
        out_root / "SETUP_B_EXACT_summary_global.csv",
        out_root / "SETUP_B_EXACT_summary_per_workload.csv",
        out_root / "causal" / "SETUP_A_feature_ranks.csv",
        out_root / "causal" / "SETUP_B_feature_ranks.csv",
        out_root / "fig5_inputs" / "setupA_edges.csv",
        out_root / "fig5_inputs" / "setupB_edges.csv",
        out_root / "figures" / "fig4_metrics_vs_window_size.png",
        out_root / "figures" / "fig6_top15_features.png",
    )
    manifest_path = write_run_manifest(
        out_root,
        repo_root=repo_root,
        data_root=Path(data_root),
        cfg=cfg,
        seed=cfg.seed,
        artifact_paths=artifact_paths,
    )

    return {
        "setup_A": df_A_z,
        "setup_B": df_B_z,
        "shared_features": shared_features,
        "model_A": model_A,
        "model_B": model_B,
        "results_A": results_A,
        "results_B": results_B,
        "ranks_A": ranks_A,
        "ranks_B": ranks_B,
        "run_manifest": manifest_path,
    }


# ---- CITADEL/TCAD ablation, now owned by this notebook ----




@dataclass(frozen=True)
class TCAD2026Config:
    scenarios_eval: tuple[str, ...] = ("DROOP", "RH", "SPECTRE")
    feature_budgets: tuple[int, ...] = (8, 15)
    window_sizes: tuple[int, ...] = (50, 100)
    lambda_res_values: tuple[float, ...] = (0.25, 0.5)
    agg_modes: tuple[str, ...] = ("max",)
    weight_modes: tuple[str, ...] = ("uniform",)
    fixed_point_q: tuple[int, ...] = (12, 15)
    n_splits: int = 3
    p_quantile: float = 0.99
    corr_threshold: float = 0.35
    seed: int = 123

    @classmethod
    def from_json(cls, path: Path) -> "TCAD2026Config":
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(
            scenarios_eval=tuple(data.get("scenarios_eval", cls.scenarios_eval)),
            feature_budgets=tuple(int(x) for x in data.get("feature_budgets", cls.feature_budgets)),
            window_sizes=tuple(int(x) for x in data.get("window_sizes", cls.window_sizes)),
            lambda_res_values=tuple(float(x) for x in data.get("lambda_res_values", cls.lambda_res_values)),
            agg_modes=tuple(data.get("agg_modes", cls.agg_modes)),
            weight_modes=tuple(data.get("weight_modes", cls.weight_modes)),
            fixed_point_q=tuple(int(x) for x in data.get("fixed_point_q", cls.fixed_point_q)),
            n_splits=int(data.get("n_splits", cls.n_splits)),
            p_quantile=float(data.get("p_quantile", cls.p_quantile)),
            corr_threshold=float(data.get("corr_threshold", cls.corr_threshold)),
            seed=int(data.get("seed", cls.seed)),
        )


def _selected_features(ranks: pd.DataFrame, k: int, fallback: Iterable[str]) -> list[str]:
    if ranks is not None and not ranks.empty and "feature" in ranks.columns:
        feats = [str(x) for x in ranks.head(int(k))["feature"].tolist()]
        if feats:
            return feats
    return list(fallback)[: int(k)]


def _fixed_point_error(df: pd.DataFrame, model, q: int, *, max_rows: int = 5000) -> dict[str, float]:
    if df.empty:
        return {"fp_mae": 0.0, "fp_max_abs": 0.0}
    subset = df.head(max_rows)
    float_score, _, _ = model.score_dataframe(subset)
    fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=int(q)))
    fixed_score = fixed.score_dataframe(subset).astype(float) / float(fixed.cfg.scale)
    diff = np.abs(float_score - fixed_score)
    return {
        "fp_mae": float(np.mean(diff)) if diff.size else 0.0,
        "fp_max_abs": float(np.max(diff)) if diff.size else 0.0,
    }


def _summarize_fold_results(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    global_df = df[df["workload"] == "ALL"].copy()
    if global_df.empty:
        global_df = df.copy()
    group_cols = [
        "setup",
        "scenario",
        "window_size",
        "agg_mode",
        "lambda_res",
        "p_quantile",
        "top_k",
        "weight_mode",
        "fixed_point_q",
        "n_selected_features",
    ]
    metric_cols = [
        "auc_roc",
        "auc_pr",
        "f1",
        "bal_acc",
        "mcc",
        "brier",
        "ece",
        "fp_mae",
        "fp_max_abs",
        "hw_frequency_ghz",
        "hw_operator_bit_width",
        "hw_std_area_mm2",
        "hw_agg_area_mm2",
        "hw_area_mm2",
        "hw_power_mw",
        "hw_setup_b_area_overhead_pct",
        "hw_idle_power_overhead_pct",
        "hw_median_workload_power_overhead_pct",
        "hw_add_count",
        "hw_mult_count",
        "hw_estimated_serial_cycles",
        "hw_add_delay_ps",
        "hw_mult_delay_ps",
    ]
    available_metrics = [c for c in metric_cols if c in global_df.columns]
    return (
        global_df.groupby(group_cols, as_index=False)[available_metrics]
        .mean()
        .sort_values(group_cols)
        .reset_index(drop=True)
    )


def run_tcad_ablation(
    *,
    data_root: Path,
    out_root: Path,
    cfg: TCAD2026Config = TCAD2026Config(),
) -> dict[str, Path | pd.DataFrame]:
    """Run the TCAD extension ablation grid.

    This is deliberately a reference orchestration layer. It reuses the ETS
    pipeline components, then varies the journal-specific design axes:
    feature budget, aggregation, decision-block length, lambda, weighting, and
    fixed-point precision.
    """
    data_root = Path(data_root)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    repo_root = REPO_ROOT

    df_a, df_b = load_telemetry_two_setups(data_root)
    df_by_setup = {
        "A": clean_and_debias_telemetry(df_a),
        "B": clean_and_debias_telemetry(df_b),
    }

    feat_a = drop_constant_features(df_by_setup["A"], get_feature_columns(df_by_setup["A"]))
    feat_b = drop_constant_features(df_by_setup["B"], get_feature_columns(df_by_setup["B"]))
    shared_features = sorted(set(feat_a) & set(feat_b))
    if not shared_features:
        raise RuntimeError("No shared numeric telemetry features between Setup A and Setup B.")

    ranks_by_setup: dict[str, pd.DataFrame] = {}
    for setup, df_setup in df_by_setup.items():
        base_model = fit_cintas_from_benign(
            df_setup,
            shared_features,
            lambda_res=0.5,
            weight_mode="uniform",
        )
        score, _, _ = base_model.score_dataframe(df_setup)
        df_setup["cias_sample_score"] = score
        _, _, ranks = build_causal_and_rank_features_for_setup(
            setup=setup,
            df=df_setup,
            feature_cols=shared_features,
            out_root=out_root,
            corr_threshold=cfg.corr_threshold,
            top_k_plot=max(cfg.feature_budgets),
            score_col="cias_sample_score",
        )
        ranks_by_setup[setup] = ranks

    fold_frames: list[pd.DataFrame] = []
    selected_rows: list[dict] = []

    for setup, df_setup in df_by_setup.items():
        for top_k, lambda_res, agg_mode, weight_mode, q in product(
            cfg.feature_budgets,
            cfg.lambda_res_values,
            cfg.agg_modes,
            cfg.weight_modes,
            cfg.fixed_point_q,
        ):
            feats = _selected_features(ranks_by_setup[setup], int(top_k), shared_features)
            hw_cost = estimate_cintas_hardware_cost(feature_names=feats, frequency_ghz=1.0)
            model = fit_cintas_from_benign(
                df_setup,
                feats,
                lambda_res=float(lambda_res),
                weight_mode=str(weight_mode),
            )
            fp_err = _fixed_point_error(df_setup, model, int(q))
            selected_rows.append({
                "setup": setup,
                "top_k": int(top_k),
                "lambda_res": float(lambda_res),
                "agg_mode": str(agg_mode),
                "weight_mode": str(weight_mode),
                "fixed_point_q": int(q),
                "n_selected_features": int(hw_cost.feature_count),
                "feature_group_counts": format_feature_group_counts(hw_cost.group_feature_counts),
                "features": ",".join(feats),
            })

            eval_df = run_exact_eval_for_setup(
                setup=setup,
                df=df_setup,
                model=model,
                scenarios_eval=cfg.scenarios_eval,
                window_sizes=cfg.window_sizes,
                n_splits_list=(cfg.n_splits,),
                default_p_quantile=cfg.p_quantile,
                agg_mode=str(agg_mode),
                droop_cfg=None,
                seed=cfg.seed,
            )
            if eval_df.empty:
                continue
            eval_df["top_k"] = int(top_k)
            eval_df["n_selected_features"] = int(hw_cost.feature_count)
            eval_df["weight_mode"] = str(weight_mode)
            eval_df["fixed_point_q"] = int(q)
            eval_df["fp_mae"] = fp_err["fp_mae"]
            eval_df["fp_max_abs"] = fp_err["fp_max_abs"]
            for key, value in hw_cost.to_summary_dict().items():
                eval_df[key] = value
            fold_frames.append(eval_df)

    fold_results = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
    summary = _summarize_fold_results(fold_results)

    fold_path = out_root / "tcad_ablation_fold_results.csv"
    summary_path = out_root / "tcad_ablation_summary.csv"
    selected_path = out_root / "tcad_selected_features.csv"
    config_path = out_root / "tcad_config_resolved.json"

    fold_results.to_csv(fold_path, index=False)
    summary.to_csv(summary_path, index=False)
    pd.DataFrame(selected_rows).to_csv(selected_path, index=False)
    config_path.write_text(json.dumps(asdict(cfg), indent=2, sort_keys=True), encoding="utf-8")

    manifest_path = write_run_manifest(
        out_root,
        repo_root=repo_root,
        data_root=data_root,
        cfg=cfg,
        seed=cfg.seed,
        artifact_paths=(
            fold_path,
            summary_path,
            selected_path,
            config_path,
            out_root / "causal" / "SETUP_A_feature_ranks.csv",
            out_root / "causal" / "SETUP_B_feature_ranks.csv",
        ),
    )

    return {
        "fold_results": fold_results,
        "summary": summary,
        "fold_path": fold_path,
        "summary_path": summary_path,
        "selected_path": selected_path,
        "manifest_path": manifest_path,
    }

SOURCE_REGISTRY = DATA_SOURCE_CONFIG
MANIFEST_ROOT = REPO_ROOT / "data" / "external_manifests"


@dataclass(frozen=True)
class SourceFile:
    repo: str
    ref: str
    source_id: str
    source_path: str
    rel_path: str
    name: str
    size: int
    git_sha: str
    raw_url: str
    target_path: Path


def load_source_registry(path: Path = SOURCE_REGISTRY) -> dict[str, dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def github_token() -> str | None:
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if token:
        return token.strip()
    try:
        proc = subprocess.run(
            ["gh", "auth", "token"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return None
    token = proc.stdout.strip()
    return token or None


def api_json(url: str, token: str | None) -> Any:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "CITADEL-data-prep",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def quote_path(path: str) -> str:
    return urllib.parse.quote(path, safe="/")


def raw_url(repo: str, ref: str, path: str) -> str:
    return f"https://raw.githubusercontent.com/{repo}/{ref}/{quote_path(path)}"


def repo_tree(repo: str, ref: str, token: str | None) -> list[dict[str, Any]]:
    url = f"https://api.github.com/repos/{repo}/git/trees/{quote_path(ref)}?recursive=1"
    payload = api_json(url, token)
    if payload.get("truncated"):
        raise RuntimeError(f"GitHub returned a truncated tree for {repo}@{ref}; use a narrower source path.")
    return payload.get("tree", [])


def source_files(source_id: str, spec: dict[str, Any], token: str | None) -> list[SourceFile]:
    repo = spec["repo"]
    ref = spec.get("ref", "main")
    root = spec["path"].strip("/")
    target_root = REPO_ROOT / spec["target"]
    preserve_tree = bool(spec.get("preserve_tree", True))
    include_prefixes = spec.get("include_prefixes")

    files: list[SourceFile] = []
    for entry in repo_tree(repo, ref, token):
        if entry.get("type") != "blob":
            continue
        path = str(entry.get("path", ""))
        if not path.startswith(root + "/"):
            continue
        rel = path[len(root) + 1 :]
        if include_prefixes and not any(rel.startswith(prefix) for prefix in include_prefixes):
            continue
        local_rel = rel if preserve_tree else Path(rel).name
        files.append(
            SourceFile(
                repo=repo,
                ref=ref,
                source_id=source_id,
                source_path=path,
                rel_path=rel,
                name=Path(rel).name,
                size=int(entry.get("size", 0)),
                git_sha=str(entry.get("sha", "")),
                raw_url=raw_url(repo, ref, path),
                target_path=target_root / local_rel,
            )
        )
    return sorted(files, key=lambda f: f.rel_path)


def split_csv_arg(value: str | None) -> set[str] | None:
    if not value:
        return None
    items = {x.strip().upper() for x in value.split(",") if x.strip()}
    return items or None


def filter_ddr_files(
    files: Iterable[SourceFile],
    setups: set[str] | None,
    scenarios: set[str] | None,
    workloads: set[str] | None,
) -> list[SourceFile]:
    selected = []
    for item in files:
        parts = Path(item.name).stem.split("_")
        if len(parts) < 3:
            continue
        setup, scenario, workload = parts[0].upper(), parts[1].upper(), parts[2].upper()
        if setups and setup not in setups:
            continue
        if scenarios and scenario not in scenarios:
            continue
        if workloads and workload not in workloads:
            continue
        selected.append(item)
    return selected


def filter_apple_tiers(files: Iterable[SourceFile], tiers: set[str] | None) -> list[SourceFile]:
    if not tiers:
        return list(files)
    selected = []
    for item in files:
        top = item.rel_path.split("/", 1)[0].upper()
        if top in tiers:
            selected.append(item)
    return selected


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_one_source_file(item: SourceFile, token: str | None, force: bool = False) -> dict[str, Any]:
    item.target_path.parent.mkdir(parents=True, exist_ok=True)
    if item.target_path.exists() and not force:
        return {
            "status": "exists",
            "sha256": sha256_file(item.target_path),
            "local_size": item.target_path.stat().st_size,
        }

    headers = {"User-Agent": "CITADEL-data-prep"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(item.raw_url, headers=headers)
    tmp_path = item.target_path.with_suffix(item.target_path.suffix + ".tmp")
    with urllib.request.urlopen(req, timeout=240) as resp, tmp_path.open("wb") as out:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    tmp_path.replace(item.target_path)
    return {
        "status": "downloaded",
        "sha256": sha256_file(item.target_path),
        "local_size": item.target_path.stat().st_size,
    }


def manifest_entry(item: SourceFile, downloaded: dict[str, Any] | None = None) -> dict[str, Any]:
    entry = {
        "source_id": item.source_id,
        "repo": item.repo,
        "ref": item.ref,
        "source_path": item.source_path,
        "relative_path": item.rel_path,
        "name": item.name,
        "size": item.size,
        "git_sha": item.git_sha,
        "raw_url": item.raw_url,
        "target_path": str(item.target_path.relative_to(REPO_ROOT)),
    }
    if downloaded:
        entry.update(downloaded)
    return entry


def write_external_manifest(
    source_id: str,
    spec: dict[str, Any],
    files: list[SourceFile],
    entries: list[dict[str, Any]],
) -> Path:
    MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    path = MANIFEST_ROOT / f"{source_id}.json"
    payload = {
        "source_id": source_id,
        "repo": spec["repo"],
        "ref": spec.get("ref", "main"),
        "source_path": spec["path"],
        "target": spec["target"],
        "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "file_count": len(files),
        "total_bytes": sum(f.size for f in files),
        "files": entries,
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path


def select_sources(registry: dict[str, dict[str, Any]], requested: str) -> list[str]:
    if requested == "all":
        return list(registry.keys())
    if requested not in registry:
        raise KeyError(f"Unknown source {requested!r}. Available: {', '.join(registry)}")
    return [requested]


def prepare_external_data_notebook(
    source: str = "all",
    *,
    download: bool = False,
    force: bool = False,
    setups: str | None = None,
    scenarios: str | None = None,
    workloads: str | None = None,
    apple_tiers: str | None = None,
    max_files: int | None = None,
) -> pd.DataFrame:
    """Create source manifests and, when requested, fetch DDR/Apple telemetry into this repo."""
    registry = load_source_registry()
    token = github_token()
    requested = select_sources(registry, source)

    setup_filter = split_csv_arg(setups)
    scenario_filter = split_csv_arg(scenarios)
    workload_filter = split_csv_arg(workloads)
    tier_filter = split_csv_arg(apple_tiers)

    rows: list[dict[str, Any]] = []
    for source_id in requested:
        spec = registry[source_id]
        files = source_files(source_id, spec, token)
        if spec.get("kind") == "hardware_counter":
            files = filter_ddr_files(files, setup_filter, scenario_filter, workload_filter)
        if spec.get("kind") == "limited_observability_host":
            files = filter_apple_tiers(files, tier_filter)
        if max_files is not None:
            files = files[: int(max_files)]

        entries: list[dict[str, Any]] = []
        for idx, item in enumerate(files, start=1):
            downloaded = download_one_source_file(item, token=token, force=force) if download else None
            entries.append(manifest_entry(item, downloaded=downloaded))
            rows.append({
                "source": source_id,
                "index": idx,
                "path": str(item.target_path.relative_to(REPO_ROOT)),
                "status": (downloaded or {}).get("status", "manifest_only"),
                "bytes": item.size,
            })

        manifest = write_external_manifest(source_id, spec, files, entries)
        rows.append({
            "source": source_id,
            "index": None,
            "path": str(manifest.relative_to(REPO_ROOT)),
            "status": "manifest_written",
            "bytes": sum(f.size for f in files),
        })
    return pd.DataFrame(rows)


def notebook_generate_sample_data(
    out_root: Path = REPO_ROOT / "data" / "sample",
    *,
    seed: int = SEED,
    rows: int = SAMPLE_ROWS,
    workloads: tuple[str, ...] = ("dft", "dj", "mm", "tr"),
) -> list[Path]:
    configure_reproducibility(seed=int(seed), threads=THREADS, matplotlib_backend="Agg")
    return create_sample_dataset(Path(out_root), seed=int(seed), workloads=tuple(workloads), n_rows=int(rows))


def notebook_run_exact_ets2026(
    *,
    data_root: Path,
    out_root: Path,
    seed: int = SEED,
    threads: int = THREADS,
    lambda_res: float = 0.5,
    agg_mode: str = "max",
    p_quantile: float = 0.99,
    corr_threshold: float = 0.35,
    window_sizes: tuple[int, ...] = (50, 100, 200),
    n_splits: int = 3,
) -> dict[str, Any]:
    configure_reproducibility(seed=int(seed), threads=int(threads), matplotlib_backend="Agg")
    cfg = ETS2026Config(
        window_sizes=tuple(int(x) for x in window_sizes),
        n_splits=int(n_splits),
        lambda_res=float(lambda_res),
        agg_mode=str(agg_mode),
        p_quantile=float(p_quantile),
        corr_threshold=float(corr_threshold),
        seed=int(seed),
    )
    return run_ets2026(data_root=Path(data_root), out_root=Path(out_root), cfg=cfg)


def notebook_tcad_config(
    preset: str = TCAD_PRESET,
    *,
    config_path: Path | None = None,
    seed: int = SEED,
) -> TCAD2026Config:
    resolved = Path(config_path) if config_path is not None else REPO_ROOT / "configs" / f"tcad_grid_{preset}.json"
    cfg = TCAD2026Config.from_json(resolved)
    return TCAD2026Config(
        scenarios_eval=cfg.scenarios_eval,
        feature_budgets=cfg.feature_budgets,
        window_sizes=cfg.window_sizes,
        lambda_res_values=cfg.lambda_res_values,
        agg_modes=cfg.agg_modes,
        weight_modes=cfg.weight_modes,
        fixed_point_q=cfg.fixed_point_q,
        n_splits=cfg.n_splits,
        p_quantile=cfg.p_quantile,
        corr_threshold=cfg.corr_threshold,
        seed=int(seed),
    )


def notebook_run_tcad_ablation(
    *,
    data_root: Path,
    out_root: Path,
    cfg: TCAD2026Config | None = None,
    preset: str = TCAD_PRESET,
    config_path: Path | None = None,
    seed: int = SEED,
    threads: int = THREADS,
) -> dict[str, Path | pd.DataFrame]:
    configure_reproducibility(seed=int(seed), threads=int(threads), matplotlib_backend="Agg")
    resolved_cfg = cfg if cfg is not None else notebook_tcad_config(preset, config_path=config_path, seed=seed)
    return run_tcad_ablation(data_root=Path(data_root), out_root=Path(out_root), cfg=resolved_cfg)


def notebook_plot_fig5_benign_causal_network(
    edges_a: Path,
    edges_b: Path,
    out_png: Path = RESULTS_ROOT / "figures" / "fig5_benign_sparse_causal_network.png",
    *,
    title_a: str = "Setup A",
    title_b: str = "Setup B",
    seed: int = 42,
) -> Path:
    configure_reproducibility(seed=int(seed), threads=THREADS, matplotlib_backend="Agg")
    plot_benign_sparse_causal_network_two_panel(
        edges_csv_A=Path(edges_a),
        edges_csv_B=Path(edges_b),
        out_png=Path(out_png),
        title_A=title_a,
        title_B=title_b,
        seed=int(seed),
    )
    return Path(out_png)


## 3. Prepare Data

This cell verifies the tracked telemetry folders used by the notebook. DDR4/DDR5 data is read from `data/telemetry/processed/ddr_data/`. Apple tier-0/1/2 data is read from `data/telemetry/raw/apple_data/` for limited-observability studies; it is not mixed into CINTAS hardware-cost claims. If `DATA_MODE` is set to `sample`, the notebook generates deterministic synthetic telemetry for a tiny pipeline check.


### Workload and Anomaly Labels

The hardware-counter experiments use the workload set $\mathcal{W}_{L}$, which contains 13 workload tags: DFT, DJ, DP, GS, GL, HA, JA, MM, NI, OE, PI, SH, and TR. These tags are inherited from the telemetry CSV filenames and are treated as stable workload labels throughout the notebook. Each workload is evaluated under benign operation and under SLM-relevant anomaly classes. The anomaly set is $\mathcal{A}=\{\mathrm{DROOP},\mathrm{RH},\mathrm{SPECTRE}\}$. DROOP represents voltage droop behavior. RH represents RowHammer/TRRespass-like memory disturbance. SPECTRE represents a speculative-execution security condition. BENIGN data is used for calibration and thresholding; anomaly labels are used only for evaluation.

| Label group | Labels | Notebook role |
|---|---|---|
| Workloads | DFT, DJ, DP, GS, GL, HA, JA, MM, NI, OE, PI, SH, TR | Repeated operating conditions used to test whether CITADEL is stable across workload behavior. |
| Benign class | BENIGN | Used to compute normalization constants, feature ranks, and thresholds. |
| Anomaly classes | DROOP, RH, SPECTRE | Used only after calibration to evaluate detection quality. |


In [ ]:
if DATA_SOURCE_CONFIG.exists():
    sources = json.loads(DATA_SOURCE_CONFIG.read_text())
    display(pd.DataFrame([
        {"source": key, "repo": val["repo"], "target": val["target"], "kind": val.get("kind", "")}
        for key, val in sources.items()
    ]))

display(pd.DataFrame([
    {
        "dataset": "ddr_data",
        "root": str(DDR_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(DDR_CSVS),
        "role": "CITADEL CINTAS experiments",
    },
    {
        "dataset": "apple_data",
        "root": str(APPLE_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(APPLE_CSVS),
        "role": "limited-observability analysis",
    },
]))

if DATA_MODE == "sample":
    created = notebook_generate_sample_data(DATA_ROOT, seed=SEED, rows=SAMPLE_ROWS)
    print(f"Generated {len(created)} deterministic telemetry CSVs under {DATA_ROOT}")
else:
    csvs = sorted(DATA_ROOT.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {DATA_ROOT}")
    expected = 78
    if len(csvs) != expected:
        print(f"Warning: expected {expected} DDR CSVs, found {len(csvs)}")
    print(f"Using {len(csvs)} DDR telemetry CSVs from {DATA_ROOT}")

sample_files = sorted(DATA_ROOT.glob("*.csv"))[:8]
display(pd.DataFrame({"example_input_files": [str(p.relative_to(REPO_ROOT)) for p in sample_files]}))


## 4. ETS-Style Baseline Reproduction

This section runs the conference-version pipeline on the selected telemetry snapshot:

1. load Setup A and Setup B telemetry
2. clean and debias telemetry
3. fit benign-only CINTAS calibration
4. build causal/ranking artifacts
5. evaluate decision-block metrics
6. write figures, CSV summaries, and a run manifest


In [ ]:
ets_artifacts = notebook_run_exact_ets2026(
    data_root=DATA_ROOT,
    out_root=ETS_OUT,
    seed=SEED,
    threads=THREADS,
    lambda_res=0.5,
    agg_mode="max",
    window_sizes=(50, 100, 200),
    n_splits=3,
)
print(f"ETS manifest: {ets_artifacts['run_manifest'].relative_to(REPO_ROOT)}")
print(f"Shared features: {len(ets_artifacts['shared_features'])}")

ets_a = pd.read_csv(ETS_OUT / "SETUP_A_EXACT_summary_global.csv")
ets_b = pd.read_csv(ETS_OUT / "SETUP_B_EXACT_summary_global.csv")
display(pd.concat([ets_a.assign(setup="A"), ets_b.assign(setup="B")], ignore_index=True).head(12))


## 5. TCAD Design-Space Ablation

This section runs the TCAD extension sweep. The smoke preset is intentionally small; the full preset expands feature budgets, decision-block sizes, score weights, aggregation choices, and fixed-point precisions.

Each summary row includes detection metrics and hardware-cost columns derived from the operator table under `hardware/`.


In [ ]:
cfg_path = REPO_ROOT / "configs" / f"tcad_grid_{TCAD_PRESET}.json"
tcad_cfg = notebook_tcad_config(TCAD_PRESET, config_path=cfg_path, seed=SEED)
print(f"TCAD config: {cfg_path.relative_to(REPO_ROOT)}")

tcad_artifacts = notebook_run_tcad_ablation(
    data_root=DATA_ROOT,
    out_root=TCAD_OUT,
    cfg=tcad_cfg,
    seed=SEED,
    threads=THREADS,
)
summary = tcad_artifacts["summary"].copy()
print(f"TCAD summary: {tcad_artifacts['summary_path'].relative_to(REPO_ROOT)}")
print(f"TCAD manifest: {tcad_artifacts['manifest_path'].relative_to(REPO_ROOT)}")
display(summary.head(10))


## 6. Reproducibility Check

This cell reruns the TCAD ablation into a second output directory and checks that the summary table is bit-for-bit identical at the DataFrame level. This is the notebook equivalent of the repo smoke test.


In [ ]:
if RUN_REPEAT_CHECK:
    repeat_artifacts = notebook_run_tcad_ablation(
        data_root=DATA_ROOT,
        out_root=TCAD_REPEAT_OUT,
        cfg=tcad_cfg,
        seed=SEED,
        threads=THREADS,
    )
    repeat_summary = repeat_artifacts["summary"].copy()
    same = summary.equals(repeat_summary)
    print(f"Repeated TCAD summary equals first run: {same}")
    if not same:
        diff_cols = [c for c in summary.columns if not summary[c].equals(repeat_summary[c])]
        raise AssertionError(f"Reproducibility check failed; differing columns: {diff_cols}")
else:
    print("RUN_REPEAT_CHECK=False, skipped repeat execution.")


## 7. Manifest And Artifact Audit

Run manifests are the reproducibility contract. They capture the config, runtime environment, package versions, input hashes, output hashes, and git commit.


In [ ]:
def load_manifest(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

ets_manifest = load_manifest(ETS_OUT / "run_manifest.json")
tcad_manifest = load_manifest(TCAD_OUT / "run_manifest.json")

manifest_overview = pd.DataFrame([
    {
        "run": "ETS baseline",
        "git_commit": ets_manifest.get("git_commit"),
        "data_files": len(ets_manifest.get("data_files", [])),
        "artifacts": len(ets_manifest.get("artifacts", [])),
        "python": ets_manifest.get("python_version"),
    },
    {
        "run": "TCAD ablation",
        "git_commit": tcad_manifest.get("git_commit"),
        "data_files": len(tcad_manifest.get("data_files", [])),
        "artifacts": len(tcad_manifest.get("artifacts", [])),
        "python": tcad_manifest.get("python_version"),
    },
])
display(manifest_overview)

display(pd.DataFrame(tcad_manifest.get("data_files", [])).head())

## 8. Hardware-Cost Summary

This section uses the hardware information you provided from Eduardo Ortega's reference calculation.

Operator source table:

- add: raw area `1165.234`, power `0.178 mW`, delay `62.7 ps`, cycles `3`
- mult: raw area `4532.164`, power `0.5146 mW`, delay `29.09 ps`, cycles `2`
- raw area is divided by `1000**2`, matching the reference calculation
- STD block per feature: `2 * mult + add`
- STD adder tree: `(n_features - 1) * add`
- AGG block: `2 * mult`
- power scales linearly with GHz


In [ ]:
from exact.hardware import OperatorCosts, compute_tableIII_setupB, estimate_cintas_hardware_cost

costs = OperatorCosts.from_csv(REPO_ROOT / "hardware" / "cintas_operator_costs.csv")
operator_table = pd.DataFrame([
    {"operator": "add", "area_mm2": costs.add_area_mm2, "power_mw_at_1ghz": costs.add_power_mw_at_1ghz, "delay_ps": costs.add_delay_ps, "cycles": costs.add_cycles},
    {"operator": "mult", "area_mm2": costs.mult_area_mm2, "power_mw_at_1ghz": costs.mult_power_mw_at_1ghz, "delay_ps": costs.mult_delay_ps, "cycles": costs.mult_cycles},
])
display(operator_table)

display(compute_tableIII_setupB(n_features=15, frequency_ghz=1.0))

hardware_cols = [
    "setup", "scenario", "top_k", "n_selected_features", "fixed_point_q",
    "hw_area_mm2", "hw_power_mw", "hw_setup_b_area_overhead_pct",
    "hw_idle_power_overhead_pct", "hw_add_count", "hw_mult_count",
]
display(summary[hardware_cols].drop_duplicates().head(12))

## 9. FPGA/RTL Laptop Workflow

You can approach FPGA work on a laptop in three layers.

### Layer A: No FPGA board required

1. Use Python to export fixed-point golden vectors.
2. Simulate RTL against those vectors.
3. Debug bit-exact arithmetic and stream timing.
4. Save simulation pass/fail summaries under `results/rtl_sweep/`.

This is enough to connect RTL correctness to the notebook.

### Layer B: Open-source synthesis on laptop

1. Install a simulator such as Verilator or Icarus Verilog.
2. Install Yosys for synthesis experiments.
3. For small open FPGA targets, add the matching place-and-route tools.
4. Run synthesis through notebook-managed cells or import the completed tool reports.
5. Write `rtl_resource_summary.csv` with LUT/FF/DSP/BRAM, timing, cycles, and estimated energy.

This gives preliminary hardware evidence, but it depends on the target family.

### Layer C: Vendor FPGA flow

1. Pick a target board and FPGA family.
2. Use the vendor toolchain for authoritative synthesis and implementation reports.
3. On macOS laptops, vendor FPGA tools are often not native; a Linux workstation, server, or VM is usually the practical path.
4. Export the reports to CSV.
5. Let this notebook merge the CSV with the TCAD ablation table.

The notebook integration path is: Python fixed-point model -> golden vectors -> RTL simulation/synthesis -> CSV report -> merged TCAD paper table.


In [ ]:
tools = []
for name, command in [
    ("verilator", ["verilator", "--version"]),
    ("iverilog", ["iverilog", "-V"]),
    ("yosys", ["yosys", "-V"]),
    ("gtkwave", ["gtkwave", "--version"]),
]:
    exe = shutil.which(command[0])
    row = {"tool": name, "available": exe is not None, "path": exe or ""}
    if exe:
        try:
            completed = subprocess.run(command, capture_output=True, text=True, timeout=10)
            first_line = (completed.stdout or completed.stderr).splitlines()[0] if (completed.stdout or completed.stderr) else ""
            row["version"] = first_line
        except Exception as exc:
            row["version"] = f"version check failed: {exc}"
    else:
        row["version"] = "not installed"
    tools.append(row)

display(pd.DataFrame(tools))

## 10. Export Fixed-Point Golden Vectors For RTL

This cell exports a small golden-vector file from the Python fixed-point CINTAS reference. The RTL testbench should stream these feature values and compare its output score against `expected_score_q`.


In [ ]:
from exact.cintas import FixedPointCINTAS, FixedPointConfig

FPGA_OUT.mkdir(parents=True, exist_ok=True)
model = ets_artifacts["model_A"]
setup_a = ets_artifacts["setup_A"]
q_format = 15
fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=q_format))
subset = setup_a[model.feature_cols].head(32).copy()
score_float, e1_float, e2_float = model.score_dataframe(subset)
score_q = fixed.score_dataframe(subset)

golden = subset.copy()
golden.insert(0, "sample_index", range(len(golden)))
golden["expected_score_q"] = score_q
golden["expected_score_float"] = score_float
golden["expected_e1_float"] = e1_float
golden["expected_e2_float"] = e2_float
golden_path = FPGA_OUT / f"cintas_setupA_q{q_format}_golden_vectors.csv"
golden.to_csv(golden_path, index=False)

print(f"Golden vectors: {golden_path.relative_to(REPO_ROOT)}")
display(golden.head())

## 11. Merge Future RTL/FPGA Results Into The TCAD Table

When simulation or synthesis is ready, save a CSV at `results/notebook_run/rtl_sweep/rtl_resource_summary.csv` or `results/rtl_sweep/rtl_resource_summary.csv` with columns like:

```text
setup,top_k,fixed_point_q,luts,ffs,dsps,brams,fmax_mhz,latency_cycles,energy_per_block_nj
```

The cell below loads that file if it exists. Otherwise it writes a template so the expected schema is clear.


In [ ]:
rtl_summary_candidates = [
    RTL_SWEEP_OUT / "rtl_resource_summary.csv",
    REPO_ROOT / "results" / "rtl_sweep" / "rtl_resource_summary.csv",
]
existing = next((p for p in rtl_summary_candidates if p.exists()), None)

if existing is None:
    RTL_SWEEP_OUT.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame([
        {
            "setup": "A",
            "top_k": int(summary["top_k"].iloc[0]),
            "fixed_point_q": int(summary["fixed_point_q"].iloc[0]),
            "luts": None,
            "ffs": None,
            "dsps": None,
            "brams": None,
            "fmax_mhz": None,
            "latency_cycles": None,
            "energy_per_block_nj": None,
            "status": "fill after RTL simulation/synthesis",
        }
    ])
    existing = RTL_SWEEP_OUT / "rtl_resource_summary.csv"
    template.to_csv(existing, index=False)
    print(f"Created RTL summary template: {existing.relative_to(REPO_ROOT)}")

rtl_summary = pd.read_csv(existing)
print(f"RTL summary source: {existing.relative_to(REPO_ROOT)}")
display(rtl_summary.head())

merge_keys = [key for key in ["setup", "top_k", "fixed_point_q"] if key in rtl_summary.columns and key in summary.columns]
if merge_keys:
    merged = summary.merge(rtl_summary, on=merge_keys, how="left")
    display(merged.head())
else:
    print("RTL summary does not yet share merge keys with the TCAD summary.")

## 12. Paper-Ready Output Checklist

After this notebook completes, use these generated artifacts to keep the journal work organized:

- ETS baseline manifest: `results/notebook_run/ets_baseline/run_manifest.json`
- TCAD ablation manifest: `results/notebook_run/tcad_ablation/run_manifest.json`
- TCAD ablation summary: `results/notebook_run/tcad_ablation/tcad_ablation_summary.csv`
- selected features and COM/MEM/SEN counts: `results/notebook_run/tcad_ablation/tcad_selected_features.csv`
- fixed-point golden vectors: `results/notebook_run/fpga/cintas_setupA_q15_golden_vectors.csv`
- RTL/FPGA result template or merged report: `results/notebook_run/rtl_sweep/rtl_resource_summary.csv`

For the final journal run, switch from deterministic sample data to the frozen real telemetry snapshot and run the full TCAD grid.


In [ ]:
final_artifacts = [
    ETS_OUT / "run_manifest.json",
    TCAD_OUT / "run_manifest.json",
    TCAD_OUT / "tcad_ablation_summary.csv",
    TCAD_OUT / "tcad_selected_features.csv",
    golden_path,
    existing,
]

for artifact in final_artifacts:
    print(f"{'OK' if artifact.exists() else 'MISSING'}  {artifact.relative_to(REPO_ROOT)}")

print("\nNotebook run complete.")